In [ ]:
import kagglehub
import os

def get_dataset(url):
    path = kagglehub.dataset_download(url)

    print("Path to dataset files:", path)

    for dirname, _, filenames in os.walk(path):
        for filename in filenames:
            print(os.path.relpath(os.path.join(dirname, filename), path))

    return path

ds_ratings_path = get_dataset("arashnic/book-recommendation-dataset")

# copy books csv
import shutil
shutil.copy(os.path.join(ds_ratings_path, "Books.csv"), ".")

Path to dataset files: /home/barto/.cache/kagglehub/datasets/arashnic/book-recommendation-dataset/versions/3
Ratings.csv
Users.csv
recsys_taxonomy2.png
classicRec.png
DeepRec.png
Books.csv


'./Books.csv'

In [102]:
import numpy as np
import pandas as pd


books_df =pd.read_csv(os.path.join(ds_ratings_path, "Books.csv"))
users_df =pd.read_csv(os.path.join(ds_ratings_path, "Users.csv"))
ratings_df =pd.read_csv(os.path.join(ds_ratings_path, "Ratings.csv"))

# remove implicit 0 ratings
ratings_df = ratings_df[ratings_df["Book-Rating"] != 0]

print("books: ", len(books_df))
print("users: ", len(users_df))
print("ratings: ", len(ratings_df))

print("Share of users who have age data: ", 1.0-users_df["Age"].isna().mean())

# cast to float
ratings_df["Book-Rating"] = ratings_df["Book-Rating"].astype(float)

# books_df.head()
# users_df.head()
# ratings_df.head()

# ratings_df["ISBN"].value_counts().plot(kind="hist", bins=50)

counts_per_book = ratings_df["ISBN"].value_counts()
print("Ratings count per book, min, max, mean: ", counts_per_book.min(), counts_per_book.max(), counts_per_book.mean())

# ratings_df["User-ID"].value_counts().plot(kind="hist", bins=50)

counts_per_user = ratings_df["User-ID"].value_counts()
print("Ratings count per user, min, max, mean: ", counts_per_user.min(), counts_per_user.max(), counts_per_user.mean())



# filter out books and users with few ratings
MIN_RATINGS = 5
book_counts = ratings_df["ISBN"].value_counts()
user_counts = ratings_df["User-ID"].value_counts()

df_book_limited = ratings_df[
    ratings_df["ISBN"].isin(book_counts[book_counts >= MIN_RATINGS].index)
]

df_user_limited = ratings_df[
    ratings_df["User-ID"].isin(user_counts[user_counts >= MIN_RATINGS].index)
]

print(f"Books with at least {MIN_RATINGS} ratings: {df_book_limited['ISBN'].nunique()}")
print(f"Users with at least {MIN_RATINGS} ratings: {df_user_limited['User-ID'].nunique()}")




counts = ratings_df["ISBN"].value_counts()
result = books_df.set_index("ISBN").reindex(counts.index)[["Book-Title", "Book-Author"]]
result["num_ratings"] = counts.values
result

# result["Book-Title"].isna().sum()

/tmp/ipykernel_9258/3899186870.py:5: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  books_df =pd.read_csv(os.path.join(ds_ratings_path, "Books.csv"))


books:  271360
users:  278858
ratings:  433671
Share of users who have age data:  0.6028014258152895
Ratings count per book, min, max, mean:  1 707 2.3319030181800584
Ratings count per user, min, max, mean:  1 8524 5.573819163292847
Books with at least 5 ratings: 14535
Users with at least 5 ratings: 14220


,Book-Title,Book-Author,num_ratings
ISBN,,,
0316666343,The Lovely Bones: A Novel,Alice Sebold,707
0971880107,Wild Animus,Rich Shapero,581
0385504209,The Da Vinci Code,Dan Brown,487
0312195516,The Red Tent (Bestselling Backlist),Anita Diamant,383
0679781587,NaN,NaN,333
...,...,...,...
0671563149,MUDDY WATER (Peter Bartholomew Mysteries),Sally Gunning,1
1575660792,Gray Matter,Shirley Kennett,1
0380796155,White Abacus,Damien Broderick,1


In [ ]:
# # create sparse matrix user-book rating
# from scipy.sparse import csr_matrix
# user_c = ratings_df["User-ID"].astype("category")
# book_c = ratings_df["ISBN"].astype("category")
# matrix = csr_matrix(
#     (ratings_df["Book-Rating"], (user_c.cat.codes, book_c.cat.codes))
# )

# from scipy.sparse.linalg import svds

# U, sigma, Vt = svds(matrix, k=50)

# print(np.sum(sigma**2) / np.sum(matrix.data**2))

MIN_RATINGS = 15

book_counts = ratings_df["ISBN"].value_counts()
# user_counts = df["User-ID"].value_counts()

ratings_df = ratings_df[
    ratings_df["ISBN"].isin(book_counts[book_counts >= MIN_RATINGS].index)
]

print(f"Books with at least {MIN_RATINGS} ratings: {len(ratings_df)}")

Books with at least 15 ratings: 115168
